# Ergebnisse: Schweizerdeutsch Ostschweiz – Visualisierungen

Dieses Notebook analysiert und visualisiert die Ergebnisse des Dialekt-Mapping-Projekts.
Es werden drei Datensätze verwendet:
- `ostschweiz_mapping_results.csv` - Kookkurrenz-Mapping IPA-Dialekt -> Hochdeutsch
- `transcriptions_clean.csv` - IPA-Transkriptionen (Referenz, Audio, SwissWhisper)
- `test_whisper_ostschweiz.csv` - Whisper-Modell-Vergleiche

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter, defaultdict
import re

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#55A868'
RED    = '#C44E52'
PURPLE = '#8172B3'

mapping = pd.read_csv('Data/ostschweiz_mapping_results.csv')
trans   = pd.read_csv('Data/transcriptions_clean.csv')
whisper = pd.read_csv('Data/test_whisper_ostschweiz.csv')

print(f'Mapping:    {mapping.shape}')
print(f'Transkript: {trans.shape}')
print(f'Whisper:    {whisper.shape}')

---
## 1. Top-20 Dialektwörter und ihre hochdeutschen Entsprechungen

In [ ]:
top20 = mapping.nlargest(20, 'Gemeinsame_Treffer').sort_values('Gemeinsame_Treffer')

unique_hg = top20['Hochdeutsch_Zuordnung'].unique()
palette = [BLUE, ORANGE, GREEN, RED, PURPLE,
           '#937860', '#DA8BC3', '#8C8C8C', '#CCB974', '#64B5CD']
color_map = {w: palette[i % len(palette)] for i, w in enumerate(unique_hg)}
bar_colors = [color_map[w] for w in top20['Hochdeutsch_Zuordnung']]

labels = [f"{row.IPA_Dialekt}  ->  {row.Hochdeutsch_Zuordnung}"
          for _, row in top20.iterrows()]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(labels, top20['Gemeinsame_Treffer'], color=bar_colors, edgecolor='white', linewidth=0.6)
ax.bar_label(bars, fmt='%d', padding=4, fontsize=9)
ax.set_xlabel('Anzahl gemeinsamer Kookkurrenzen')
ax.set_title('Top 20 haeufigste Dialektwörter mit hochdeutscher Zuordnung\n(Ostschweiz IPA-Audio)')
ax.set_xlim(0, top20['Gemeinsame_Treffer'].max() * 1.15)

legend_patches = [mpatches.Patch(color=color_map[w], label=w) for w in unique_hg]
ax.legend(handles=legend_patches, title='Hochdeutsches Wort',
          bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

---
## 2. Wie viele Dialektvarianten hat jedes hochdeutsche Wort?

Zeigt, welche hochdeutschen Wörter besonders viele verschiedene Dialektformen haben.

In [ ]:
variants = (mapping.groupby('Hochdeutsch_Zuordnung')
                   .agg(Varianten=('IPA_Dialekt', 'count'),
                        Gesamttreffer=('Gemeinsame_Treffer', 'sum'))
                   .reset_index()
                   .sort_values('Varianten', ascending=False)
                   .head(15))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars1 = axes[0].bar(variants['Hochdeutsch_Zuordnung'], variants['Varianten'],
                    color=BLUE, edgecolor='white')
axes[0].bar_label(bars1, padding=2, fontsize=9)
axes[0].set_ylabel('Anzahl Dialektvarianten')
axes[0].set_title('Hochdeutsche Wörter mit den meisten\nDialektvarianten (Top 15)')
axes[0].tick_params(axis='x', rotation=45)

bars2 = axes[1].bar(variants['Hochdeutsch_Zuordnung'], variants['Gesamttreffer'],
                    color=ORANGE, edgecolor='white')
axes[1].bar_label(bars2, padding=2, fontsize=9)
axes[1].set_ylabel('Summe der Kookkurrenzen')
axes[1].set_title('Gesamthaeufigkeit im Korpus\n(Summe aller Variantentreffer)')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Variationsreichtum pro hochdeutschem Wort', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Alle Dialektvarianten für ausgewählte hochdeutsche Wörter

In [ ]:
top_words = variants.head(6)['Hochdeutsch_Zuordnung'].tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
colors_cycle = [BLUE, ORANGE, GREEN, RED, PURPLE, '#937860']

for i, word in enumerate(top_words):
    subset = (mapping[mapping['Hochdeutsch_Zuordnung'] == word]
              .sort_values('Gemeinsame_Treffer', ascending=True))
    ax = axes[i]
    bars = ax.barh(subset['IPA_Dialekt'], subset['Gemeinsame_Treffer'],
                   color=colors_cycle[i], edgecolor='white')
    ax.bar_label(bars, padding=3, fontsize=8)
    ax.set_title(f'{word} - {len(subset)} IPA-Variante(n)', fontsize=11)
    ax.set_xlabel('Kookkurrenzen')
    ax.tick_params(axis='y', labelsize=10)

plt.suptitle('Dialektvarianten pro hochdeutschem Wort (Ostschweiz)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. IPA-Phonem-Frequenz: Dialekt vs. Referenz

Vergleich der häufigsten IPA-Zeichen in der gesprochenen Dialektaufnahme (`ipa_audio`) gegenüber der hochdeutschen Referenztranskription (`ipa_reference`).

In [ ]:
ost = trans[trans['dialect_region'] == 'Ostschweiz'].copy()

def count_ipa_chars(series):
    text = ' '.join(series.dropna().astype(str))
    return Counter(c for c in text if c not in (' ', '\t', '\n'))

ref_counts   = count_ipa_chars(ost['ipa_reference'])
audio_counts = count_ipa_chars(ost['ipa_audio'])

all_chars = set(list(ref_counts.keys()) + list(audio_counts.keys()))
top_chars = sorted(all_chars,
                   key=lambda c: ref_counts.get(c, 0) + audio_counts.get(c, 0),
                   reverse=True)[:25]

ref_vals   = [ref_counts.get(c, 0)   for c in top_chars]
audio_vals = [audio_counts.get(c, 0) for c in top_chars]

x = np.arange(len(top_chars))
w = 0.38

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, ref_vals,   width=w, color=BLUE,   label='IPA Referenz (Hochdeutsch)',         alpha=0.85)
ax.bar(x + w/2, audio_vals, width=w, color=ORANGE, label='IPA Audio (Ostschweizer Dialekt)',    alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top_chars, fontsize=12)
ax.set_ylabel('Häufigkeit')
ax.set_title('Top-25 IPA-Zeichen: Referenztranskription vs. Dialektaufnahme (Ostschweiz)')
ax.legend()
plt.tight_layout()
plt.show()

diff_df = pd.DataFrame({'Zeichen': top_chars, 'Referenz': ref_vals, 'Audio': audio_vals})
diff_df['Differenz'] = diff_df['Audio'] - diff_df['Referenz']
print('\nZeichen mit groesster Abweichung (Audio - Referenz):')
print(diff_df.reindex(diff_df['Differenz'].abs().sort_values(ascending=False).index).head(10).to_string(index=False))

---
## 5. Wortanzahl pro Satz: Dialekt vs. Referenz

Wie viele IPA-Tokens hat ein Satz im Dialekt im Vergleich zur hochdeutschen Referenz?

In [ ]:
ost = ost.copy()
ost['n_ref']   = ost['ipa_reference'].fillna('').apply(lambda s: len(s.split()))
ost['n_audio'] = ost['ipa_audio'].fillna('').apply(lambda s: len(s.split()))
ost['n_swiss'] = ost['ipa_swiss_whisper'].fillna('').apply(lambda s: len(s.split()))
ost['diff']    = ost['n_audio'] - ost['n_ref']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

bins = range(0, 25)
axes[0].hist(ost['n_ref'],   bins=bins, alpha=0.7, color=BLUE,   label='IPA Referenz', edgecolor='white')
axes[0].hist(ost['n_audio'], bins=bins, alpha=0.7, color=ORANGE, label='IPA Audio',    edgecolor='white')
axes[0].axvline(ost['n_ref'].median(),   color=BLUE,   linestyle='--', lw=1.5,
                label=f'Median Ref: {ost["n_ref"].median():.1f}')
axes[0].axvline(ost['n_audio'].median(), color=ORANGE, linestyle='--', lw=1.5,
                label=f'Median Audio: {ost["n_audio"].median():.1f}')
axes[0].set_xlabel('Anzahl IPA-Tokens pro Satz')
axes[0].set_ylabel('Anzahl Saetze')
axes[0].set_title('Tokenanzahl pro Satz')
axes[0].legend(fontsize=9)

diff_counts = ost['diff'].value_counts().sort_index()
axes[1].bar(diff_counts.index, diff_counts.values, color=GREEN, edgecolor='white')
axes[1].axvline(0, color='black', lw=1.2, linestyle='--')
axes[1].set_xlabel('Tokendifferenz (Audio - Referenz)')
axes[1].set_ylabel('Anzahl Saetze')
axes[1].set_title('Differenz der Tokenanzahl\n(>0 = Dialekt hat mehr Tokens)')

axes[2].hist(ost['n_ref'],   bins=bins, alpha=0.7, color=BLUE,   label='IPA Referenz',     edgecolor='white')
axes[2].hist(ost['n_swiss'], bins=bins, alpha=0.7, color=PURPLE, label='IPA SwissWhisper', edgecolor='white')
axes[2].set_xlabel('Anzahl IPA-Tokens pro Satz')
axes[2].set_title('Tokenanzahl: SwissWhisper vs. Referenz')
axes[2].legend(fontsize=9)

plt.suptitle('Satzlaenge in IPA-Tokens', fontsize=13)
plt.tight_layout()
plt.show()

print(ost[['n_ref', 'n_audio', 'n_swiss']].describe().round(2))

---
## 6. IPA-Wortlängen: Dialekt vs. Referenz-Tokens

In [ ]:
def token_lengths(series):
    tokens = []
    for s in series.dropna().astype(str):
        tokens.extend(len(w) for w in s.split() if w)
    return tokens

ref_lens   = token_lengths(ost['ipa_reference'])
audio_lens = token_lengths(ost['ipa_audio'])
swiss_lens = token_lengths(ost['ipa_swiss_whisper'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bins2 = range(1, 18)
axes[0].hist(ref_lens,   bins=bins2, alpha=0.7, color=BLUE,   density=True, edgecolor='white',
             label=f'Referenz (Mittel: {np.mean(ref_lens):.2f})')
axes[0].hist(audio_lens, bins=bins2, alpha=0.7, color=ORANGE, density=True, edgecolor='white',
             label=f'Audio (Mittel: {np.mean(audio_lens):.2f})')
axes[0].hist(swiss_lens, bins=bins2, alpha=0.6, color=PURPLE, density=True, edgecolor='white',
             label=f'SwissWhisper (Mittel: {np.mean(swiss_lens):.2f})')
axes[0].set_xlabel('Laenge des IPA-Tokens (Zeichen)')
axes[0].set_ylabel('Relative Haeufigkeit')
axes[0].set_title('Verteilung der IPA-Wortlaengen')
axes[0].legend(fontsize=9)

bp = axes[1].boxplot([ref_lens, audio_lens, swiss_lens],
                     labels=['Referenz', 'Audio', 'SwissWhisper'],
                     patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], [BLUE, ORANGE, PURPLE]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_ylabel('Tokenlaenge (Zeichen)')
axes[1].set_title('Boxplot: IPA-Wortlaengen im Vergleich')

plt.suptitle('IPA-Wortlaengen: Referenz vs. Dialektaudio vs. SwissWhisper', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Whisper-Modell-Vergleich: Character Error Rate (CER)

Wir berechnen die **CER** zwischen dem Referenzsatz und den verschiedenen Whisper-Ausgaben. Niedrigere CER = bessere Transkription.

In [ ]:
def cer(ref, hyp):
    ref = str(ref).lower().strip()
    hyp = str(hyp).lower().strip()
    if len(ref) == 0:
        return 0.0 if len(hyp) == 0 else 1.0
    m, n = len(ref), len(hyp)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            if ref[i-1] == hyp[j-1]:
                dp[j] = prev[j-1]
            else:
                dp[j] = 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[n] / len(ref)

models = {
    'Whisper v2 HD':   'A_v2_hd',
    'SwissWhisper HD': 'B_swiss_hd',
    'Wav2Vec IPA':     'C_w2v_ipa',
    'Whisper IPA':     'D_whisper_ipa',
}

cer_results = {}
for name, col in models.items():
    cer_results[name] = [cer(ref, hyp)
                         for ref, hyp in zip(whisper['sentence'], whisper[col])]

cer_df = pd.DataFrame(cer_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bp = axes[0].boxplot(cer_df.values,
                     labels=cer_df.columns,
                     patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
colors_models = [BLUE, GREEN, ORANGE, RED]
for patch, c in zip(bp['boxes'], colors_models):
    patch.set_facecolor(c)
    patch.set_alpha(0.75)
axes[0].set_ylabel('Character Error Rate (CER)')
axes[0].set_title('CER pro Transkriptionsmodell')
axes[0].tick_params(axis='x', rotation=15)

mean_cer = cer_df.mean().sort_values()
bar_colors_m = [colors_models[list(cer_df.columns).index(c)] for c in mean_cer.index]
bars = axes[1].bar(mean_cer.index, mean_cer.values, color=bar_colors_m,
                   edgecolor='white', alpha=0.85)
axes[1].bar_label(bars, fmt='%.3f', padding=3, fontsize=10)
axes[1].set_ylabel('Mittlere CER')
axes[1].set_title('Mittlere CER pro Modell (niedriger = besser)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].set_ylim(0, mean_cer.max() * 1.2)

plt.suptitle('Vergleich der Whisper-Modelle: Character Error Rate', fontsize=13)
plt.tight_layout()
plt.show()

print('\nMittlere CER pro Modell:')
print(cer_df.mean().sort_values().to_string())

---
## 8. CER-Verteilung: Violinplot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

parts = ax.violinplot(cer_df.values,
                      positions=range(len(cer_df.columns)),
                      showmedians=True, showextrema=True)

for body, c in zip(parts['bodies'], colors_models):
    body.set_facecolor(c)
    body.set_alpha(0.6)
parts['cmedians'].set_color('black')
parts['cmedians'].set_linewidth(2)

ax.set_xticks(range(len(cer_df.columns)))
ax.set_xticklabels(cer_df.columns, rotation=15)
ax.set_ylabel('Character Error Rate (CER)')
ax.set_title('Vollstaendige CER-Verteilung pro Modell (Violinplot)')

means = cer_df.mean().values
ax.scatter(range(len(means)), means, color='black', zorder=5, s=50, label='Mittelwert')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 9. Kookkurrenz-Heatmap: Top-12 IPA x Top-12 Hochdeutsch

Welche IPA-Dialektwörter sind besonders stark mit welchen hochdeutschen Wörtern assoziiert?

In [ ]:
ost_full = trans[trans['dialect_region'] == 'Ostschweiz'].copy()

word_mapping = defaultdict(Counter)
for _, row in ost_full.iterrows():
    hg_words  = re.sub(r'[^\w\s]', '', str(row['sentence']).lower()).split()
    ipa_words = str(row['ipa_audio']).split()
    for iw in ipa_words:
        if len(iw) > 1:
            for hw in hg_words:
                word_mapping[iw][hw] += 1

top_ipa = sorted(word_mapping.keys(),
                 key=lambda k: sum(word_mapping[k].values()),
                 reverse=True)[:12]

all_hg = Counter()
for k in top_ipa:
    all_hg.update(word_mapping[k])
top_hg = [w for w, _ in all_hg.most_common(12)]

matrix = np.array([[word_mapping[i].get(h, 0) for h in top_hg] for i in top_ipa], dtype=float)
row_sums = matrix.sum(axis=1, keepdims=True)
matrix_norm = np.divide(matrix, row_sums, where=row_sums > 0)

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(matrix_norm, cmap='YlOrRd', aspect='auto', vmin=0, vmax=matrix_norm.max())

ax.set_xticks(range(len(top_hg)))
ax.set_xticklabels(top_hg, rotation=40, ha='right', fontsize=11)
ax.set_yticks(range(len(top_ipa)))
ax.set_yticklabels(top_ipa, fontsize=11)
ax.set_xlabel('Hochdeutsches Wort')
ax.set_ylabel('Ostschweizer IPA-Dialektwort')
ax.set_title('Kookkurrenz-Heatmap (zeilenweise normalisiert)\nTop-12 IPA x Top-12 Hochdeutsch')

for i in range(len(top_ipa)):
    for j in range(len(top_hg)):
        val = matrix_norm[i, j]
        if val > 0.05:
            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                    fontsize=8, color='black' if val < 0.6 else 'white')

plt.colorbar(im, ax=ax, label='Relativer Anteil (normalisiert)')
plt.tight_layout()
plt.show()

---
## 10. Zusammenfassung der wichtigsten Befunde

In [ ]:
best_model    = cer_df.mean().idxmin()
best_cer_val  = cer_df.mean().min()
most_var_word = variants.iloc[0]['Hochdeutsch_Zuordnung']
most_var_n    = int(variants.iloc[0]['Varianten'])
top1_ipa      = mapping.nlargest(1, 'Gemeinsame_Treffer').iloc[0]

print('=' * 55)
print('  WICHTIGSTE BEFUNDE')
print('=' * 55)
print(f'  Datensatz:              {len(ost_full)} Ostschweizer Saetze')
print(f'  Analysierte IPA-Typen:  {len(word_mapping)}')
print()
print(f'  Haeufigste Mapping:')
print(f'    IPA "{top1_ipa.IPA_Dialekt}" -> "{top1_ipa.Hochdeutsch_Zuordnung}"'
      f'  ({int(top1_ipa.Gemeinsame_Treffer)} Treffer)')
print()
print(f'  Variationsreichstes HD-Wort:')
print(f'    "{most_var_word}" hat {most_var_n} verschiedene Dialektvarianten')
print()
print(f'  Bestes Whisper-Modell:')
print(f'    {best_model}  (mittlere CER: {best_cer_val:.3f})')
print('=' * 55)